# Моделирование рекомендательной системы

**Задача:** implicit feedback, ранжирование — для каждого пользователя вернуть топ-K товаров,
которые он с наибольшей вероятностью добавит в корзину.

**Модели:**
1. Popularity Baseline — топ-K самых добавляемых в корзину товаров
2. ALS (Alternating Least Squares) — матричная факторизация с implicit feedback

**Метрики:** recall@10, NDCG@10, precision@10

**Train/val split:** по времени — последние 14 дней → holdout

In [1]:
import os
import warnings
from pathlib import Path

import implicit
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
import seaborn as sns

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (12, 5)
sns.set_style("whitegrid")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR   = Path(os.getenv("DATA_DIR",   PROJECT_ROOT / "data" / "raw"))
MODELS_DIR = Path(os.getenv("MODELS_DIR", PROJECT_ROOT / "models"))
MODELS_DIR.mkdir(parents=True, exist_ok=True)

import sys
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from features import InteractionMatrix
from models import ModelArtifact

RANDOM_STATE = 42
K = 10
print("Библиотеки загружены")

C:\Users\denis\Projects\Study\ml-ecommerce-recsys\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Библиотеки загружены


## 1. Загрузка данных и train/val split

In [2]:
events = pd.read_csv(DATA_DIR / "events.csv")
events["dt"] = pd.to_datetime(events["timestamp"], unit="ms")
events["visitorid"] = events["visitorid"].astype(int)
events["itemid"]    = events["itemid"].astype(int)

# Временной сплит: последние 14 дней → holdout
cutoff = events["dt"].max() - pd.Timedelta(days=14)
train_events = events[events["dt"] < cutoff].copy()
val_events   = events[events["dt"] >= cutoff].copy()

print(f"Train: {len(train_events):,} событий")
print(f"  Период: {train_events['dt'].min().date()} — {train_events['dt'].max().date()}")
print(f"\nVal:   {len(val_events):,} событий")
print(f"  Период: {val_events['dt'].min().date()} — {val_events['dt'].max().date()}")

# Таргет валидации: addtocart в holdout-периоде
val_cart = val_events[val_events["event"] == "addtocart"]
val_users_with_cart = val_cart["visitorid"].unique()
print(f"\nПользователей с addtocart в val: {len(val_users_with_cart):,}")

Train: 2,509,138 событий
  Период: 2015-05-03 — 2015-09-04

Val:   246,963 событий
  Период: 2015-09-04 — 2015-09-18

Пользователей с addtocart в val: 3,535


## 2. Построение матрицы взаимодействий

In [3]:
matrix = InteractionMatrix()
matrix.fit(train_events)

print(f"Матрица взаимодействий:")
print(f"  Пользователей: {matrix.n_users:,}")
print(f"  Товаров:        {matrix.n_items:,}")
print(f"  Ненулевых элементов: {matrix.user_item.nnz:,}")
print(f"  Заполненность (density): {matrix.density:.6%}")

# Распределение суммарных весов по пользователям
user_weights = np.array(matrix.user_item.sum(axis=1)).flatten()
print(f"\nВеса пользователей: медиана={np.median(user_weights):.1f}, "
      f"макс={user_weights.max():.0f}")

Матрица взаимодействий:
  Пользователей: 1,280,329
  Товаров:        225,427
  Ненулевых элементов: 1,950,423
  Заполненность (density): 0.000676%

Веса пользователей: медиана=1.0, макс=14164


## 3. Функции оценки

In [4]:
def build_val_ground_truth(val_cart, item_map):
    """Построить dict {visitorid: set(item_idx)} для оценки."""
    gt = {}
    for _, row in val_cart.iterrows():
        uid  = int(row["visitorid"])
        iid  = int(row["itemid"])
        iidx = item_map.get(iid)
        if iidx is None:
            continue
        gt.setdefault(uid, set()).add(iidx)
    return gt


def recall_at_k(recs_idx, gt_idx, k):
    hits = len(set(recs_idx[:k]) & gt_idx)
    return hits / len(gt_idx) if gt_idx else 0.0


def ndcg_at_k(recs_idx, gt_idx, k):
    dcg  = sum(1.0 / np.log2(i + 2) for i, r in enumerate(recs_idx[:k]) if r in gt_idx)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(gt_idx), k)))
    return dcg / idcg if idcg > 0 else 0.0


def precision_at_k(recs_idx, gt_idx, k):
    hits = len(set(recs_idx[:k]) & gt_idx)
    return hits / k


def evaluate(get_recs_fn, gt, k=10, max_users=1000):
    """Вычислить recall, NDCG, precision@k по выборке пользователей."""
    recalls, ndcgs, precisions = [], [], []
    users = list(gt.keys())[:max_users]
    for uid in users:
        recs = get_recs_fn(uid)
        if not recs:
            continue
        gt_set = gt[uid]
        recalls.append(recall_at_k(recs, gt_set, k))
        ndcgs.append(ndcg_at_k(recs, gt_set, k))
        precisions.append(precision_at_k(recs, gt_set, k))
    return {
        f"recall@{k}": float(np.mean(recalls)),
        f"ndcg@{k}": float(np.mean(ndcgs)),
        f"precision@{k}": float(np.mean(precisions)),
        "n_evaluated": len(recalls),
    }


gt = build_val_ground_truth(val_cart, matrix.item_map)
print(f"Пользователей с known ground truth: {len(gt):,}")

Пользователей с known ground truth: 3,426


## 4. Popularity Baseline

In [5]:
# Популярность: число уникальных пользователей, добавивших товар в корзину (в train)
train_cart = train_events[train_events["event"] == "addtocart"]
popular_items = (
    train_cart.groupby("itemid")["visitorid"]
    .nunique()
    .sort_values(ascending=False)
)
popular_items_idx = [
    matrix.item_map[iid]
    for iid in popular_items.index
    if iid in matrix.item_map
]

print(f"Товаров с ≥1 addtocart в train: {len(popular_items):,}")
print(f"Из них присутствуют в матрице:   {len(popular_items_idx):,}")


def popularity_recs(uid, n=K):
    """Вернуть топ-n популярных товаров, которые пользователь ещё не видел."""
    if uid in matrix.user_map:
        user_idx = matrix.user_map[uid]
        seen = set(matrix.user_item[user_idx].indices)
    else:
        seen = set()
    return [idx for idx in popular_items_idx if idx not in seen][:n]


metrics_pop = evaluate(popularity_recs, gt, k=K, max_users=2000)
print("\nPopularity Baseline:")
for k, v in metrics_pop.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

Товаров с ≥1 addtocart в train: 22,367
Из них присутствуют в матрице:   22,367

Popularity Baseline:
  recall@10: 0.0150
  ndcg@10: 0.0098
  precision@10: 0.0024
  n_evaluated: 2000


## 5. ALS — Alternating Least Squares

In [ ]:
als_model = implicit.als.AlternatingLeastSquares(
    factors=64,
    iterations=20,
    regularization=0.05,
    random_state=RANDOM_STATE,
    use_gpu=False,
    num_threads=4,
)

print("Обучаем ALS...")
# implicit >= 0.7: fit() принимает (n_users × n_items)
als_model.fit(matrix.user_item)
print("Готово!")

n_uf = als_model.user_factors.shape[0]
n_if = als_model.item_factors.shape[0]
print(f"\nMatrix : {matrix.n_users:,} users × {matrix.n_items:,} items")
print(f"ALS    : {n_uf:,} user_factors × {n_if:,} item_factors")
assert n_uf == matrix.n_users, f"Несоответствие users: {n_uf} != {matrix.n_users}"
assert n_if == matrix.n_items, f"Несоответствие items: {n_if} != {matrix.n_items}"
print("Размеры согласованы ✓")


def als_recs(uid, n=K):
    """ALS-рекомендации; cold-start → popularity fallback.

    implicit 0.7+:
      fit()       → user_item (n_users × n_items)
      recommend() → одна строка user_item[user_idx] (1 × n_items)
    """
    if uid not in matrix.user_map:
        return popular_items_idx[:n]
    user_idx = matrix.user_map[uid]
    user_row = matrix.user_item[user_idx]   # (1 × n_items)
    ids, _ = als_model.recommend(
        user_idx,
        user_row,
        N=n,
        filter_already_liked_items=True,
    )
    return list(ids)


metrics_als = evaluate(als_recs, gt, k=K, max_users=2000)
print("\nALS (factors=64, iter=20):")
for key, v in metrics_als.items():
    print(f"  {key}: {v:.4f}" if isinstance(v, float) else f"  {key}: {v}")

## 6. Сравнение моделей

In [ ]:
comparison = pd.DataFrame({
    "Popularity Baseline": metrics_pop,
    "ALS (factors=64)": metrics_als,
}).T

comparison["recall@10 vs baseline"] = (
    (comparison["recall@10"] / comparison["recall@10"].iloc[0] - 1) * 100
).map("{:+.1f}%".format)

print(comparison.to_string())

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, metric in zip(axes, [f"recall@{K}", f"ndcg@{K}", f"precision@{K}"]):
    vals = [metrics_pop[metric], metrics_als[metric]]
    bars = ax.bar(["Popularity", "ALS"], vals, color=["steelblue", "seagreen"])
    ax.bar_label(bars, fmt="{:.4f}", padding=3)
    ax.set_title(metric)
    ax.set_ylim(0, max(vals) * 1.3)

plt.tight_layout()
plt.show()

## 7. Анализ рекомендаций ALS

In [ ]:
# Пример рекомендаций для нескольких пользователей
sample_users = list(gt.keys())[:5]

print("Примеры рекомендаций ALS:")
for uid in sample_users:
    recs = als_recs(uid)
    gt_set = gt[uid]
    hits = set(recs) & gt_set
    rec_items = [int(matrix.item_ids[r]) for r in recs]
    print(f"\n  user={uid}")
    print(f"    Рекомендации (itemid): {rec_items}")
    print(f"    Ground truth (idx):    {sorted(gt_set)}")
    print(f"    Hits: {len(hits)} / recall@10={recall_at_k(recs, gt_set, K):.2f}")

In [ ]:
# Распределение recall@10 по пользователям
recall_vals = []
for uid, gt_set in list(gt.items())[:2000]:
    recs = als_recs(uid)
    recall_vals.append(recall_at_k(recs, gt_set, K))

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(recall_vals, bins=20, color="seagreen", edgecolor="white")
ax.axvline(np.mean(recall_vals), color="red", linestyle="--",
           label=f"Среднее: {np.mean(recall_vals):.3f}")
ax.set_title(f"Распределение recall@{K} по пользователям (ALS)")
ax.set_xlabel(f"recall@{K}")
ax.set_ylabel("Пользователей")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Пользователей с recall@{K}=0: {(np.array(recall_vals)==0).mean():.1%}")
print(f"Пользователей с recall@{K}>0: {(np.array(recall_vals)>0).mean():.1%}")

## 8. Сохранение артефактов

In [ ]:
# Popularity baseline artifact
artifact_pop = ModelArtifact(
    model_type="popularity",
    top_k=K,
    user_map=matrix.user_map,
    item_map=matrix.item_map,
    item_ids=matrix.item_ids,
    popular_items=[int(matrix.item_ids[idx]) for idx in popular_items_idx[:200]],
    n_users=matrix.n_users,
    n_items=matrix.n_items,
    metrics=metrics_pop,
)
artifact_pop.save(MODELS_DIR / "artifact_popularity.pkl")
print("Артефакт popularity сохранён")

# ALS model + artifact
joblib.dump(als_model, MODELS_DIR / "als_model.pkl")

artifact_als = ModelArtifact(
    model_type="als",
    top_k=K,
    user_map=matrix.user_map,
    item_map=matrix.item_map,
    item_ids=matrix.item_ids,
    popular_items=[int(matrix.item_ids[idx]) for idx in popular_items_idx[:200]],
    n_users=matrix.n_users,
    n_items=matrix.n_items,
    metrics=metrics_als,
)
artifact_als.save(MODELS_DIR / "artifact.pkl")

print("ALS модель сохранена")
print(f"\nФайлы в {MODELS_DIR}:")
for f in MODELS_DIR.iterdir():
    print(f"  {f.name} ({f.stat().st_size / 1024**2:.1f} МБ)")